## 1 - Imports

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

## 2 - Loading Data
The dataset used in this notebook was generated from PostgreSQL churn metrics exported from the subscription analytics database. Each row represents a customer subscription record with aggregated payment and churn-related features.

In [5]:
from pathlib import Path

data_path = Path("churn_metric_isolation") / "database-churn-metrics.csv"
df = pd.read_csv(data_path)

## 3 - Initial Table Inspection

In [6]:
df.head()

,customer_id,plan_type,billing_cycle,total_revenue,total_payments,successful_payments,failed_payments,started_date,ended_date,subscription_length_days,churned
0,1,basic,monthly,80.0,4,4,0,2025-09-29 07:41:48+13,NaN,93,0
1,2,basic,monthly,240.0,12,11,1,2025-01-22 20:10:28+13,NaN,343,0
2,3,pro,monthly,50.0,1,1,0,2025-05-03 13:18:01+12,2025-05-08 22:56:08+12,5,1
3,4,basic,monthly,160.0,8,7,1,2025-05-11 20:06:04+12,NaN,234,0
4,5,basic,monthly,80.0,4,4,0,2025-09-11 17:56:24+12,NaN,111,0


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               1000 non-null   int64  
 1   plan_type                 1000 non-null   str    
 2   billing_cycle             1000 non-null   str    
 3   total_revenue             1000 non-null   float64
 4   total_payments            1000 non-null   int64  
 5   successful_payments       1000 non-null   int64  
 6   failed_payments           1000 non-null   int64  
 7   started_date              1000 non-null   str    
 8   ended_date                158 non-null    str    
 9   subscription_length_days  1000 non-null   int64  
 10  churned                   1000 non-null   int64  
dtypes: float64(1), int64(6), str(4)
memory usage: 86.1 KB


In [8]:
df['churned'].value_counts(normalize=True)
print(f"{df['churned'].value_counts(normalize=True)[0]*100}% of subscribers do not churn, while {df['churned'].value_counts(normalize=True)[1]*100}% do")

84.2% of subscribers do not churn, while 15.8% do


The dataset is imbalanced, with churned customers forming a minority of the records. This means accuracy alone may be misleading, so recall, precision, and the confusion matrix will be important evaluation metrics.

## 4 - Dropping Columns to Create `df_simple`

The `customer_id` column is removed because it is an identifier rather than a predictive feature. The raw `started_date` and `ended_date` columns are also removed at this stage because they are timestamp fields and are already partly represented by the engineered `subscription_length_days` feature. Additionally, `ended_date` should be treated carefully because it may encode information about the target variable directly.

In [9]:
df_simple = df.drop(columns=[
    'customer_id',
    'started_date',
    'ended_date'
]).copy()

In [10]:
df_simple.head()

,plan_type,billing_cycle,total_revenue,total_payments,successful_payments,failed_payments,subscription_length_days,churned
0,basic,monthly,80.0,4,4,0,93,0
1,basic,monthly,240.0,12,11,1,343,0
2,pro,monthly,50.0,1,1,0,5,1
3,basic,monthly,160.0,8,7,1,234,0
4,basic,monthly,80.0,4,4,0,111,0


In [11]:
df_simple.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   plan_type                 1000 non-null   str    
 1   billing_cycle             1000 non-null   str    
 2   total_revenue             1000 non-null   float64
 3   total_payments            1000 non-null   int64  
 4   successful_payments       1000 non-null   int64  
 5   failed_payments           1000 non-null   int64  
 6   subscription_length_days  1000 non-null   int64  
 7   churned                   1000 non-null   int64  
dtypes: float64(1), int64(5), str(2)
memory usage: 62.6 KB


## 5 - Label Encoding

We first need to encode the string values used in `billing_cycle` and `plan_type` into numerical values that the models will be able to understand.
<br><br>
Because `billing_cycle` and `plan_type` are binary categorical features in this dataset, label encoding is acceptable for this first model. However, for categorical variables with more than two categories, one-hot encoding would usually be preferred to avoid imposing an artificial ordering.

In [12]:
### Identifies string variable (e.g. monthly and yearly) and encodes them to integers (monthly->0, yearly->1)
billing_encoder = LabelEncoder()
plan_encoder = LabelEncoder()

df_simple['billing_cycle'] = billing_encoder.fit_transform(df_simple['billing_cycle'])
df_simple['plan_type'] = plan_encoder.fit_transform(df_simple['plan_type'])

In [13]:
print(billing_encoder.classes_)
print(f"{billing_encoder.classes_[0]} = 0\n{billing_encoder.classes_[1]} = 1\n")
print(plan_encoder.classes_)
print(f"{plan_encoder.classes_[0]} = 0\n{plan_encoder.classes_[1]} = 1")


['monthly' 'yearly']
monthly = 0
yearly = 1

['basic' 'pro']
basic = 0
pro = 1


Here is the new dataframe with the encoded values

In [14]:
df_simple.head()

,plan_type,billing_cycle,total_revenue,total_payments,successful_payments,failed_payments,subscription_length_days,churned
0,0,0,80.0,4,4,0,93,0
1,0,0,240.0,12,11,1,343,0
2,1,0,50.0,1,1,0,5,1
3,0,0,160.0,8,7,1,234,0
4,0,0,80.0,4,4,0,111,0


# 6 - First Model

## 6.1 - Train / Test Split

Since we are only dealing with a single dataset of 1000 subscribers, we need to seperate the data into a training set and a test set. For this project we will make this split 80% train and 20% test.
<br>
The initial split will include the following features:
- Plan Type
- Billing Cycle
- Total Payments
- Successful Payments
- Failed Payments


In [15]:
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    df_simple[['plan_type', 'billing_cycle', 'total_payments',
               'successful_payments', 'failed_payments']],
    df_simple['churned'],
    test_size=0.2,
    random_state=42,
    stratify=df_simple['churned']
)

## 6.2 - Creating Model

Using `class_weight='balanced'` helps the model pay more attention to churned customers, but it can also increase false positives. In a churn context, this may be acceptable if identifying at-risk customers is more important than avoiding false alarms. Attempts to create models without this weighting resulted in the model only ever predicting no churn, because this guarenteed an 84.2% accuracy.

In [16]:
model_1 = LogisticRegression(max_iter=1000, class_weight='balanced')


In [17]:
model_1.fit(X_train_1, y_train_1)
y_pred_1 = model_1.predict(X_test_1)

## 6.3 - Model Evaluation

### 6.3.1 - Prediction Evaluation

In [18]:
print(f"Accuracy: {accuracy_score(y_test_1, y_pred_1)}")
print(classification_report(y_test_1, y_pred_1))
print(confusion_matrix(y_test_1, y_pred_1))

Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.92      0.57      0.70       168
           1       0.25      0.75      0.37        32

    accuracy                           0.59       200
   macro avg       0.58      0.66      0.54       200
weighted avg       0.81      0.59      0.65       200

[[95 73]
 [ 8 24]]


Accuracy = 59.5%
<br>
Class 1 Precision = 25%
<br>
Class 1 Recall = 75%
<br>
Confusion Matrix:
<br>

| | Predicted Churn | Predicted No Churn|
|-|-|-|
Churn|95|73|
No Churn| 8|24|

### 6.3.2 - Coefficient Analysis

In [19]:
coefficients_1 = pd.DataFrame({
    'Feature': X_train_1.columns,
    'Coefficients': model_1.coef_[0]
})

print(coefficients_1)

               Feature  Coefficients
0            plan_type     -0.209861
1        billing_cycle     -3.456201
2       total_payments     -0.008078
3  successful_payments     -0.188602
4      failed_payments      0.180524


| Feature | Coefficient | Interpretation & Analysis |
| -------- | -------- |-------- |
| Plan Type  | -0.21   | The model finds a weak negative association between plan type and churn. With `pro = 1`, this suggests that pro subscribers are slightly less likely to churn than basic subscribers. |
| Billing Cycle | -3.46 | Billing cycle has a strong negative association with churn. With `yearly = 1`, the model predicts that yearly subscribers are associated with a substantially lower likelihood of churn than monthly subscribers. This likely reflects fewer renewal opportunities and the structure of the dataset. |
| Total Payments | -0.008 | The model finds almost no association between total payments and churn. This may be because total payments is strongly related to billing cycle and subscription length, meaning its independent contribution is limited. |
| Successful Payments | -0.19| The model associates more successful payments with lower churn, but this feature is highly correlated with total payments and failed payments, so its coefficient should not be interpreted in isolation. |
| Failed Payments | +0.18 | The model associates more failed payments with higher churn probability. However, because failed payments are strongly related to total payments and successful payments, this coefficient reflects a shared relationship between these features rather than a fully independent effect. |


The coefficients above suggest that `total_payments` has very little effect on the model’s prediction. However, this may be misleading. Because `successful_payments` and `failed_payments` are both directly derived from `total_payments`, these features are highly correlated. This multicollinearity makes it difficult for the model to isolate the independent effect of `total_payments`, potentially understating its true association with churn..
<br>
<br>
TP = SP + FP


# 7 - Second Model (+ PSR & PFR)

## 7.05 - Feature Engineering

In order to free up `total_payments` to test this relationship, the following change will be applied:
- Removing `succesful_payments` and `failed_payments` as features, and instead replacing them with `payment_success_rate` and `payment_fail_rate`. These are more useful metrics to use, because they take into account the number of payments a subscriber has made. e.g. A subscriber on a yearly billing cycle failing 1 payment is more significant than a monthly subscriber failing 1 payment.

In [20]:
df_simple['payment_success_rate'] = (
    df_simple['successful_payments']
    / df_simple['total_payments']
)

df_simple['payment_failure_rate'] = (
    df_simple['failed_payments']
    / df_simple['total_payments']
)


## 7.1 - Train / Test Split

This next split will include the following features:
- Plan Type
- Billing Cycle
- Total Payments
- Payment Success Rate (PSR)
- Payment Fail Rate (PFR)


In [21]:
### Seperate the dataset into a training set and a test set (80 train, 20 test)
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    df_simple[['plan_type', 'billing_cycle', 'total_payments',
        'payment_success_rate', 'payment_failure_rate']],
    df_simple['churned'],
    test_size=0.2,
    random_state=42,
    stratify=df_simple['churned']
)

## 7.2 - Creating Model

In [22]:
model_2 = LogisticRegression(max_iter=1000, class_weight='balanced')

In [23]:
model_2.fit(X_train_2, y_train_2)
y_pred_2 = model_2.predict(X_test_2)

## 7.3 - Model Evaluation

### 7.3.1 - Feature Evaluation

In [24]:
print(f"Accuracy: {accuracy_score(y_test_2, y_pred_2)}")
print(classification_report(y_test_2, y_pred_2))
print(confusion_matrix(y_test_2, y_pred_2))

Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.91      0.57      0.70       168
           1       0.24      0.72      0.36        32

    accuracy                           0.59       200
   macro avg       0.58      0.65      0.53       200
weighted avg       0.81      0.59      0.65       200

[[96 72]
 [ 9 23]]


Accuracy = 59.5%
<br>
Class 1 Precision = 24%
<br>
Class 1 Recall = 72%
<br>
Confusion Matrix:
<br>

| | Predicted Churn | Predicted No Churn|
|-|-|-|
Churn|96|72|
No Churn| 9|23|

### 7.3.2 - Coefficient Evaluation

In [25]:
coefficients_2 = pd.DataFrame({
    'Feature': X_train_2.columns,
    'Coefficients': model_2.coef_[0]
})

print(coefficients_2)

                Feature  Coefficients
0             plan_type     -0.216582
1         billing_cycle     -3.452964
2        total_payments     -0.178006
3  payment_success_rate     -0.250440
4  payment_failure_rate      0.251082


| Feature | Coefficient | Interpretation & Analysis |
| -------- | -------- |-------- |
| Plan Type  | -0.216582 | This coefficient value is almost identical to the previous model. This indicates that the change to `success / fail rates` hasn't had a big impact on the models association between `plan type` and `churn`|
| Billing Cycle | -3.452964 | This coefficient value has also remained almost identical to the previous model. This indicates that the change to `PSR / PFR` hasn't had a big impact on the models association between `billing cycle` and `churn` |
| Total Payments | -0.178006 | The magnitude of the `total_payments` coefficient increases substantially compared to the previous model. This suggests that replacing raw `successful / failed payment` counts with payment rates allows the model to isolate the effect of `total payments` more effectively.|
| PSR | -0.250440 | The model forms a negative association bettween `PSR` and churn. This means the higher the `PSR`, the less likely the model is to predict a subscription churning|
| PFR | +0.251082 | The model forms an almost exactly opposite association for `PFR` and `churn` compared to `PSR` and `churn`. This is because `PSR` and `PSR` are directly proportional..|

The values of `PSR` and `PFR` are directly dependent on eachother through the following relationship.

$$
PSR + PFR = 1
$$

$$
\therefore
PSR = 1 - PFR
\qquad AND \qquad
PFR = 1 - PSR
$$

Due to this relationship, these two features represent effectively the same information, and so including both makes one of them redundant. \
We will prove that next.

# 8 - Second Model Revised (PSR vs PFR)

Since we have strong reason to believe that including both `PSR` and `PFR` in a model makes one of them redundant, we will adjust the model to to only include one. \
We will test 2 models. 

## 8.1 - Train / Test Split
1. PSR only
- Plan Type
- Billing Cycle
- Total Payments
- PSR
2. PFR only
- Plan Type
- Billing Cycle
- Total Payments
- PFR

In [45]:
# PSR model train / test split
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    df_simple[['plan_type', 'billing_cycle', 'total_payments', 'payment_success_rate']],
    df_simple['churned'],
    test_size=0.2,
    random_state=42,
    stratify=df_simple['churned']
)

# PFR model train / test split
X_train_4, X_test_4, y_train_4, y_test_4 = train_test_split(
    df_simple[['plan_type', 'billing_cycle', 'total_payments', 'payment_failure_rate']],
    df_simple['churned'],
    test_size=0.2,
    random_state=42,
    stratify=df_simple['churned']
)

## 8.2 - Creating Models

In [46]:
# PSR model
model_3 = LogisticRegression(max_iter=1000, class_weight='balanced')
# PFR model
model_4 = LogisticRegression(max_iter=1000, class_weight='balanced')

In [47]:
# PSR model training
model_3.fit(X_train_3, y_train_3)
y_pred_3 = model_3.predict(X_test_3)

# PFR model training
model_4.fit(X_train_4, y_train_4)
y_pred_4 = model_4.predict(X_test_4)

## 8.3 - Model Evaluation

### 8.3.1 - Feature Evaluation

##### PSR Model

In [48]:
print(f"Accuracy: {accuracy_score(y_test_3, y_pred_3)}")
print(classification_report(y_test_3, y_pred_3))
print(confusion_matrix(y_test_3, y_pred_3))


Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.91      0.57      0.70       168
           1       0.24      0.72      0.36        32

    accuracy                           0.59       200
   macro avg       0.58      0.65      0.53       200
weighted avg       0.81      0.59      0.65       200

[[96 72]
 [ 9 23]]


##### PFR Model

In [49]:
print(f"Accuracy: {accuracy_score(y_test_4, y_pred_4)}")
print(classification_report(y_test_4, y_pred_4))
print(confusion_matrix(y_test_4, y_pred_4))

Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.91      0.57      0.70       168
           1       0.24      0.72      0.36        32

    accuracy                           0.59       200
   macro avg       0.58      0.65      0.53       200
weighted avg       0.81      0.59      0.65       200

[[96 72]
 [ 9 23]]


### 8.3.2 - Coefficient Evaluation

In [50]:
# Create coefficient tables
coefficients_3 = pd.DataFrame({
    'Feature': X_train_3.columns,
    'Coefficients': model_3.coef_[0] 
})
coefficients_4 = pd.DataFrame({
    'Feature': X_train_4.columns,
    'Coefficients': model_4.coef_[0] 
})
print("PSR Model")
print(coefficients_3)
print("\nPFR Model")
print(coefficients_4)

PSR Model
                Feature  Coefficients
0             plan_type     -0.218546
1         billing_cycle     -3.451557
2        total_payments     -0.177917
3  payment_success_rate     -0.443118

PFR Model
                Feature  Coefficients
0             plan_type     -0.218267
1         billing_cycle     -3.452475
2        total_payments     -0.178033
3  payment_failure_rate      0.439700


As expected, the resulting models behave identically in their predictions, and have almost identical coefficient values. For this reason, we will only use the `PFR` version of this model, as the result is effectively identical to the `PSR + PFR` and `PSR` versions.

# 9 - Third Model (+ Subscription Length)

We will now add a new feature, `subscription length`, which is a record of the number of days a subscriber has been subscribed. \


## 9.1 - Train / Test Split
This model has the following features:
- Plan Type
- Billing Cycle
- Total Payments
- Payment Fail Rate
- Subscription Length

In [51]:
X_train_5, X_test_5, y_train_5, y_test_5 = train_test_split(
    df_simple[['plan_type', 'billing_cycle', 'total_payments',
        'payment_failure_rate', 'subscription_length_days']],
    df_simple['churned'],
    test_size=0.2,
    random_state=42,
    stratify=df_simple['churned']
)

## 9.2 - Creating Model

In [53]:
model_5 = LogisticRegression(max_iter=1000, class_weight='balanced')

model_5.fit(X_train_5, y_train_5)
y_pred_5 = model_5.predict(X_test_5)

## 9.3 - Model Evaluation

### 9.3.1 - Feature Evaluation

In [55]:
print(f"Accuracy: {accuracy_score(y_test_5, y_pred_5)}")
print(classification_report(y_test_5, y_pred_5))
print(confusion_matrix(y_test_5, y_pred_5))

Accuracy: 0.595
              precision    recall  f1-score   support

           0       0.94      0.55      0.70       168
           1       0.26      0.81      0.39        32

    accuracy                           0.59       200
   macro avg       0.60      0.68      0.54       200
weighted avg       0.83      0.59      0.65       200

[[93 75]
 [ 6 26]]


### 9.3.2 - Coefficient Evaluation

In [54]:
coefficients_5 = pd.DataFrame({
    'Feature': X_train_5.columns,
    'Coefficients': model_5.coef_[0] 
})
print(coefficients_5)

                    Feature  Coefficients
0                 plan_type     -0.233548
1             billing_cycle     -2.339567
2            total_payments      0.310271
3      payment_failure_rate      0.478616
4  subscription_length_days     -0.016270


In [56]:
print(coefficients_4)

                Feature  Coefficients
0             plan_type     -0.218267
1         billing_cycle     -3.452475
2        total_payments     -0.178033
3  payment_failure_rate      0.439700


| Feature | Coefficient | Interpretation & Analysis |
| -------- | -------- |-------- |
| Plan Type  | -0.233548 | Similar to the previous model, the negative coefficient indicates that the model associates pro subscribers with a lower likelihood of `churn` than basic subscribers. |
| Billing Cycle | -2.339567 | The magnitude of this coefficient decreases substantially compared to previous models. This suggests that some of the predictive information previously captured by `billing cycle` is now being captured by `subscription length`, indicating that the two features contain overlapping information related to customer tenure and retention.|
| Total Payments | +0.310271 | The coefficient changes sign compared to earlier models, suggesting that the interpretation of `total payments` is highly dependent on which related features are included. This indicates that total payments may be acting as a proxy for multiple underlying behaviours rather than having a simple independent relationship with `churn`.|
| PFR | +0.478616 | The value of this coefficient is very similar to previous models. This suggests that `payment failure rate` continues to provide largely the same predictive information even after `subscription length` is introduced. |
| Sub Length | -0.016270 | The model associates longer `subscription length` with a lower likelihood of `churn`. Although the coefficient appears small, `subscription length` ranges from approximately 1–365 days, meaning small coefficient changes can accumulate into a substantial effect. The relatively small coefficient magnitude may therefore be misleading and motivates the use of feature scaling in the next section. |

The coefficients above are not directly comparable because the features have very different value ranges. \
For example, `billing_cycle` takes values of 0–1, while `subscription_length_days` ranges from roughly 1–365. \
Standard scaling transforms each feature to a common scale (mean = 0, standard deviation = 1), allowing the coefficient magnitudes to better reflect the relative influence of each feature.


# 10 - Scaling

## 10.1 - Creating the Scaler

StandardScaler transforms each feature according to:

z = (x - μ) / σ

where:
- x is the original value
- μ is the feature mean
- σ is the feature standard deviation

This centers each feature around 0 and scales it to units of standard deviations. After scaling, coefficient magnitudes become more directly comparable because all features are measured on a similar scale.

In [ ]:
scaler_1 = StandardScaler()
scaler_4 = StandardScaler()
scaler_5 = StandardScaler()

# 10.2 - Scaling the Models

In [ ]:
# Successful / Failed Payments Model
X_train_1_scaled = scaler_1.fit_transform(X_train_1)
X_test_1_scaled = scaler_1.transform(X_test_1)

# Payment Fail Rate Model
X_train_4_scaled = scaler_4.fit_transform(X_train_4)
X_test_4_scaled = scaler_4.transform(X_test_4)

# Subscription Length Model
X_train_5_scaled = scaler_5.fit_transform(X_train_5)
X_test_5_scaled = scaler_5.transform(X_test_5)

## 10.3 - Creating Scaled Models

In [ ]:
# Scaled Successful / Failed Payments Model
model_6 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_6.fit(X_train_1_scaled, y_train_1)
y_pred_6 = model_6.predict(X_test_1_scaled)

# Scaled Payment Fail Rate Model
model_7 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_7.fit(X_train_4_scaled, y_train_4)
y_pred_7 = model_7.predict(X_test_4_scaled)

# Scaled Subscription Length Model
model_8 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_8.fit(X_train_5_scaled, y_train_5)
y_pred_8 = model_8.predict(X_test_5_scaled)



In [76]:
# Scaled Successful / Failed Payments Model
coefficients_6 = pd.DataFrame({
    'Feature': X_train_1.columns,
    'Coefficients': model_6.coef_[0]
})

# Scaled Payment Fail Rate Model
coefficients_7 = pd.DataFrame({
    'Feature': X_train_4.columns,
    'Coefficients': model_7.coef_[0]
})

# Scaled Subscription Length Model
coefficients_8 = pd.DataFrame({
    'Feature': X_train_5.columns,
    'Coefficients': model_8.coef_[0]
})

print("\nScaled Successful / Failed Payments Model")
print(coefficients_6)
print("\nScaled PFR Model")
print(coefficients_7)
print("\nScaled Subscription Length Model")
print(coefficients_8)


Scaled Successful / Failed Payments Model
               Feature  Coefficients
0            plan_type     -0.092350
1        billing_cycle     -1.620803
2       total_payments     -0.293772
3  successful_payments     -0.319548
4      failed_payments      0.114335

Scaled PFR Model
                Feature  Coefficients
0             plan_type     -0.094566
1         billing_cycle     -1.620828
2        total_payments     -0.563699
3  payment_failure_rate     -0.090390

Scaled Subscription Length Model
                    Feature  Coefficients
0                 plan_type     -0.097796
1             billing_cycle     -1.270854
2            total_payments      0.302060
3      payment_failure_rate      0.094738
4  subscription_length_days     -0.861980


## 10.4 - Unscaled vs Scaled Model Comparisons

##### Unscaled vs Scaled Successful / Failed Payments Model
|Feature| Unscaled | Scaled|Explanation|
|-------|----------|-------|-----------|
|Plan Type|-0.209861|-0.092350|The coefficient decreases because plan type already exists on a small 0–1 scale, so scaling has little impact on its relative importance|
|Billing Cycle|-3.456201|-1.620803|The coefficient decreases substantially because it was previously inflated by being compared against features with much larger value ranges|
|Total Payments|-0.008078|-0.293772|The coefficient magnitude increases significantly, indicating that its influence was previously understated due to its larger numerical range|
|Successful Payments|-0.293772|-0.319548|The coefficient changes very little, suggesting that its apparent influence was already reasonably represented before scaling|
|Failed Payments|+0.180524|+0.114335|The coefficient decreases slightly after scaling, indicating that some of its apparent influence was due to differences in feature scale rather than predictive power alone|

Comparing the unscaled and scaled coefficients reveals that billing cycle remains the strongest predictor of churn, even after correcting for differences in feature scale. However, the dominance of billing cycle is far less extreme than the unscaled coefficients suggested. The coefficients for total payments and successful payments become much more comparable after scaling, indicating that these features have a larger influence on the model than initially appeared. This demonstrates that raw coefficient magnitudes can be misleading when features operate on different numerical ranges and highlights the importance of scaling before comparing feature influence.

##### Unscaled vs Scaled PFR Model
|Feature| Unscaled | Scaled|Explanaition|
|-------|----------|-------|------------|
|Plan Type|-0.218267|-0.094566|The coefficient changes very little after scaling, indicating that plan type already operated on a small numerical scale and its relative influence remains minimal|
|Billing Cycle|-3.452475|-1.620828|The coefficient decreases substantially after scaling, showing that its unscaled value overstated its relative influence due to differences in feature scale|
|Total Payments|-0.178033|-0.563699|The coefficient magnitude increases significantly after scaling, suggesting that its influence was previously understated by its larger numerical range|
|Payment Fail Rate|+0.439700|-0.090390|The coefficient decreases dramatically after scaling, indicating that much of its apparent importance in the unscaled model was caused by scale differences rather than strong independent predictive power|

The most notable finding from this model is the dramatic reduction in the apparent importance of payment failure rate after scaling. Before scaling, PFR appeared to be one of the strongest predictors in the model, but after scaling its coefficient becomes relatively small compared to billing cycle and total payments. This suggests that much of the apparent influence of PFR in the unscaled model was caused by differences in feature scale rather than genuine predictive power. Billing cycle remains the strongest predictor, while total payments emerges as a more influential feature than initially suggested.

##### Unscaled vs Scaled Subscription Length Model
|Feature| Unscaled | Scaled|Explanation|
|-------|----------|-------|-----------|
|Plan Type|-0.233548|-0.097796|The coefficient decreases slightly after scaling, indicating that plan type contributes relatively little independent information to the model|
|Billing Cycle|-2.339567|-1.270854|The coefficient decreases substantially after scaling, showing that its unscaled value overstated its relative influence due to differences in feature scale|
|Total Payments|+0.310271|+0.302060|The coefficient changes very little after scaling, suggesting that its apparent influence was already reasonably represented in the unscaled model|
|Payment Fail Rate|+0.478616|+0.094738|The coefficient decreases dramatically after scaling, indicating that much of its apparent importance in the unscaled model was driven by differences in feature scale|
|Subscription Length|-0.016270|-0.861980|The coefficient magnitude increases dramatically after scaling, revealing that subscription length has a much larger influence on churn prediction than the unscaled coefficient suggested|


This model provides the clearest demonstration of why feature scaling is necessary. Prior to scaling, subscription length appeared to have almost no influence on churn due to its very small coefficient. However, after scaling, subscription length becomes one of the most influential features in the model, with a coefficient magnitude approaching that of billing cycle. This reveals that the original coefficient was suppressed by the large numerical range of the feature (1–365 days) rather than a lack of predictive value. The scaled model therefore shows that both billing cycle and subscription length are major contributors to churn prediction, with billing cycle remaining slightly stronger. \
\
We now see that subscription length is actually one of the strongest predictors in the model and has a level of influence comparable to billing cycle. This reveals that the tiny unscaled coefficient was primarily a consequence of subscription length being measured on a much larger numerical range (1–365 days) rather than a lack of predictive power.

## 10.5 Takeaway

Although all three models achieved similar predictive performance, the Subscription Length Model was selected because it incorporates customer tenure, introduces a meaningful additional behavioural signal, and avoids the feature redundancy present in the Successful/Failed Payments model.


# 11 - Limitations & Future Work

## Limitations
- Dataset is synthetically generated
- Relationships may reflect assumptions in the data generator
- Subscription length may not be available in the same form in a real-world prediction setting
- Small dataset (1000 rows)
- Logistic regression assumes linear relationships
\
## Future Work
- One-Hot Encoding
- Cross-validation
- Random Forest
- Gradient Boosting / XGBoost
- Real-world churn dataset
- ROC-AUC evaluation